In [1]:
import numpy as np
import matplotlib.pyplot as plt
import itertools
import random
import time
from revised_greedy_up_down_functions import *

### Helper Functions

In [2]:
def get_sqdist(vec1, vec2):
    
    length = len(vec1)
    assert(len(vec2)==length)
    sum_sq = 0
    for i in range(length):
        sum_sq += (vec1[i]-vec2[i])**2
    return sum_sq

def get_shape_reps(shapes_reps, cluster):
    
    cluster_reps = []
    for i in cluster:
        cluster_reps.append(shapes_reps[i])
    return cluster_reps


def get_dist_nn_ml(best_cluster_reps, node_rep):
    
    '''This function returns the smallest distance of the query node with the best cluster'''
    least_dist = 100000000
    for i in range(len(best_cluster_reps)):
        dist = get_sqdist(node_rep, best_cluster_reps[i])
        if(dist < least_dist):
            least_dist = dist
    return least_dist

def get_dist_knn_ml(knn, best_cluster_reps, node_rep):
    
    '''This function returns the avg of the K smallest distance of the query node with the best cluster'''
    if(knn < 1 or knn > len(best_cluster_reps)):
        print("Choose an appropriate K-NN value -> [1, number of nodes in best cluster]")
        return -1
    
    least_dist = []
    for i in range(len(best_cluster_reps)):
        least_dist.append(get_sqdist(node_rep, best_cluster_reps[i]))
    least_dist = np.sort(np.array(least_dist))
    least_distance = np.sum(least_dist[:knn])/knn
    return least_distance

def get_diff_bw_clusters(a,b):
    
    for node in a:
        if(node not in b):
            print(node, "Only in First")
    for node in b:
        if(node not in a):
            print(node, "Only in Second")

In [3]:
def get_fitness_nearestNeighbour(population, dist):
    pop_size = len(population)
    fitness_vec = []
    for pop in range(pop_size):
        a = list(population[pop])
        length = len(a)
        total_distance = 0
        for i in range(length):
#             print("\nNEW ELEMENT")
            cur_closest_distance = 100000000
            for j in range(length):
                if(i!=j):
                    cur_distance = dist[a[i]][a[j]]
#                     print(cur_distance)
                    if(cur_distance < cur_closest_distance):
                        cur_closest_distance = cur_distance
#                         print("Closest distance updated to", cur_closest_distance)
#             print("FINAL CLOSEST DISTANCE FOR THE NEW ELEMENT", cur_closest_distance)
            total_distance += cur_closest_distance
#             print("NEW TOTAL DISTANCE", total_distance)
#         print("\nFINAL TOTAL DISTANCE", total_distance)
#         print("LENGTH", length)
#         print(length/total_distance)
        fitness_vec.append(length/total_distance) ## inverse of avg nearest neighbour distance
    return fitness_vec

def get_fitness_KnearestNeighbour(population, dist, knn):
    pop_size = len(population)
    fitness_vec = []
    for pop in range(pop_size):
        a = list(population[pop])
        length = len(a)
        if(knn > (length-1) or knn<1):
            print("Choose an appropriate K-NN value -> [1, number of roots - 1]")
            return -1
        total_distance = 0
        for i in range(length):
            cur_distance = []
            for j in range(length):
                if(i!=j):
                    cur_distance.append(dist[a[i]][a[j]])
#             print("\nNEW ELEMENT")
#             print(cur_distance)
#             print("SORTING....")
            cur_distance = np.sort(np.array(cur_distance))
#             print(cur_distance)
            cur_closest_distance = np.sum(cur_distance[:knn])/knn
#             print("AVERAGE CLOSEST DISTANCE", cur_closest_distance)
            total_distance += cur_closest_distance
#             print("TOTAL DISTANCE TILL NOW", total_distance)
#         print("LENGTH", length)
#         print(length/total_distance)
        fitness_vec.append(length/total_distance) ## inverse of avg nearest neighbour distance
    return fitness_vec

### Main function

In [4]:
def get_best_cluster_in_other(knn, roots, subchildren, shape_reps, best_cluster_reps, print_flag):
    
    nodes = range(np.sort(np.array(list(subchildren.keys())))[-1]+1)
    nodes_reps = get_shape_reps(shape_reps, nodes)
    
    dist_vec = []
    for node_rep in nodes_reps:
        
        if(knn == 1):
            dist_vec.append(get_dist_nn_ml(best_cluster_reps, node_rep))
        else:
            dist_vec.append(get_dist_knn_ml(knn, best_cluster_reps, node_rep))
    
    
    cur_best_cluster = []
    
    if(print_flag): print("ROOTS:", roots)
    
    for root in roots:
        
        children = subchildren[root]
        
        if(print_flag): print("\n")
        if(print_flag): print("NEW ROOT:", root, children)
        
        best_score = dist_vec[root]
        best_cluster = [root]
        
        if(print_flag): print("STARTING:", best_score, best_cluster)
        
        for child in children:
            sum_child = 0.0
            for i in range(len(child)):
                cur_score = dist_vec[child[i]]
                sum_child += cur_score
            sum_child /= len(child)
            
            if(print_flag): print("ITERATION:", child, sum_child)
            
            if(sum_child < best_score):
                best_score = sum_child
                best_cluster = list(child)
                if(print_flag): print("BEST UPDATED:", best_score, best_cluster)
                
        for node in best_cluster:
            cur_best_cluster.append(node)
    
    if(print_flag): print("\nCUR BEST CLUSTER:", cur_best_cluster)
    return cur_best_cluster

## p7

In [5]:
shapes_reps_1 = np.load("p7_ShapeReps_246x6144.npy")
shapes_dist_euc1 = np.load("p7_ShapeDist_6144_Euc.npy")

In [6]:
tree1 = dict()

for i in range(10):
    tree1[i] = dict()
    tree1[i]["child"] = [i+34]
    tree1[i]["parent"] = []
    
tree1[10] = {"child":[44, 45], "parent":[]}

for i in range(11, 27):
    tree1[i] = dict()
    tree1[i]["child"] = [i+35]
    tree1[i]["parent"] = []

tree1[27] = {"child":[62, 63], "parent":[]}

for i in range(28, 34):
    tree1[i] = dict()
    tree1[i]["child"] = [i+36]
    tree1[i]["parent"] = []

for i in range(34, 44):
    tree1[i] = dict()
    tree1[i]["child"] = [i+36]
    tree1[i]["parent"] = [i-34]

tree1[44] = {"child":[80], "parent":[10]}
tree1[45] = {"child":[], "parent":[10]}

for i in range(46, 62):
    tree1[i] = dict()
    tree1[i]["child"] = [i+35]
    tree1[i]["parent"] = [i-35]

tree1[62] = {"child":[97], "parent":[27]}
tree1[63] = {"child":[], "parent":[27]}

tree1[64] = {"child":[98], "parent":[28]}
tree1[65] = {"child":[], "parent":[29]}

for i in range(66, 70):
    tree1[i] = dict()
    tree1[i]["child"] = [i+33]
    tree1[i]["parent"] = [i-36]
    
for i in range(70, 80):
    tree1[i] = dict()
    tree1[i]["child"] = [i+33]
    tree1[i]["parent"] = [i-36]
    
tree1[80] = {"child":[113], "parent":[44]}

for i in range(81, 97):
    tree1[i] = dict()
    tree1[i]["child"] = [i+33]
    tree1[i]["parent"] = [i-35]
    
tree1[97] = {"child":[130], "parent":[62]}
tree1[98] = {"child":[131], "parent":[64]}

for i in range(99, 103):
    tree1[i] = dict()
    tree1[i]["child"] = [i+33]
    tree1[i]["parent"] = [i-33]
    
for i in range(103, 123):
    tree1[i] = dict()
    tree1[i]["child"] = [i+33]
    tree1[i]["parent"] = [i-33]
    
tree1[123] = {"child":[], "parent":[90]}

for i in range(124, 136):
    tree1[i] = dict()
    tree1[i]["child"] = [i+32]
    tree1[i]["parent"] = [i-33]

for i in range(136, 156):
    tree1[i] = dict()
    tree1[i]["child"] = [i+32]
    tree1[i]["parent"] = [i-33]

tree1[156] = {"child":[188], "parent":[124]}
tree1[157] = {"child":[], "parent":[125]}
    
for i in range(158, 163):
    tree1[i] = dict()
    tree1[i]["child"] = [i+31]
    tree1[i]["parent"] = [i-32]
    
tree1[163] = {"child":[], "parent":[131]}
tree1[164] = {"child":[], "parent":[132]}
tree1[165] = {"child":[194], "parent":[133]}
tree1[166] = {"child":[195], "parent":[134]}
tree1[167] = {"child":[], "parent":[135]}

for i in range(168, 175):
    tree1[i] = dict()
    tree1[i]["child"] = [i+28]
    tree1[i]["parent"] = [i-32]

tree1[175] = {"child":[203, 204], "parent":[143]}

for i in range(176, 189):
    tree1[i] = dict()
    tree1[i]["child"] = [i+29]
    tree1[i]["parent"] = [i-32]
    
tree1[189] = {"child":[218], "parent":[158]}
tree1[190] = {"child":[], "parent":[159]}
tree1[191] = {"child":[219], "parent":[160]}
tree1[192] = {"child":[220], "parent":[161]}
tree1[193] = {"child":[], "parent":[162]}
tree1[194] = {"child":[], "parent":[165]}
tree1[195] = {"child":[], "parent":[166]}

for i in range(196, 203):
    tree1[i] = dict()
    tree1[i]["child"] = [i+25]
    tree1[i]["parent"] = [i-28]
    
tree1[203] = {"child":[228], "parent":[175]}
tree1[204] = {"child":[229], "parent":[175]}

for i in range(205, 213):
    tree1[i] = dict()
    tree1[i]["child"] = [i+25]
    tree1[i]["parent"] = [i-29]
    
tree1[213] = {"child":[238, 239], "parent":[184]}

for i in range(214, 218):
    tree1[i] = dict()
    tree1[i]["child"] = [i+26]
    tree1[i]["parent"] = [i-29]

tree1[218] = {"child":[], "parent":[189]}
tree1[219] = {"child":[244], "parent":[191]}
tree1[220] = {"child":[245], "parent":[192]}

for i in range(221, 238):
    tree1[i] = dict()
    tree1[i]["child"] = []
    tree1[i]["parent"] = [i-25]
    
tree1[238] = {"child":[], "parent":[213]}
tree1[239] = {"child":[], "parent":[213]}

for i in range(240, 244):
    tree1[i] = dict()
    tree1[i]["child"] = []
    tree1[i]["parent"] = [i-26]
    
tree1[244] = {"child":[], "parent":[219]}
tree1[245] = {"child":[], "parent":[220]}

    
# Verifying if the tree is correct

## Print all leaf nodes
leafs1 = []
for key in tree1:
    if(len(tree1[key]["child"])==0):
        leafs1.append(key)
print("Leaf nodes:", leafs1)

## Print all at the lowest threshold nodes
roots1 = []
for key in tree1:
    if(len(tree1[key]["parent"])==0):
        roots1.append(key)
print("Root nodes:", roots1)

## Print all paths
paths1 = []
for node in leafs1:
    node_path = [node]
    cur_node = node
    while(True):
        if(len(tree1[cur_node]["parent"])!=0):
            node_path.append(tree1[cur_node]["parent"][0])
            cur_node = tree1[cur_node]["parent"][0]
            continue
        else:
            break
    paths1.append(node_path[::-1])
print("Paths:", paths1)

Leaf nodes: [45, 63, 65, 123, 157, 163, 164, 167, 190, 193, 194, 195, 218, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245]
Root nodes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]
Paths: [[10, 45], [27, 63], [29, 65], [20, 55, 90, 123], [22, 57, 92, 125, 157], [28, 64, 98, 131, 163], [30, 66, 99, 132, 164], [33, 69, 102, 135, 167], [24, 59, 94, 127, 159, 190], [27, 62, 97, 130, 162, 193], [31, 67, 100, 133, 165, 194], [32, 68, 101, 134, 166, 195], [23, 58, 93, 126, 158, 189, 218], [0, 34, 70, 103, 136, 168, 196, 221], [1, 35, 71, 104, 137, 169, 197, 222], [2, 36, 72, 105, 138, 170, 198, 223], [3, 37, 73, 106, 139, 171, 199, 224], [4, 38, 74, 107, 140, 172, 200, 225], [5, 39, 75, 108, 141, 173, 201, 226], [6, 40, 76, 109, 142, 174, 202, 227], [7, 41, 77, 110, 143, 175, 203, 228], [7, 41, 77, 110, 143, 175, 204, 229], [8, 42, 78, 1

In [7]:
subchildren1 = get_complete_children_sets(tree1)

## p46

In [8]:
shapes_reps_2 = np.load("p46_ShapeReps_198x6144.npy")
shapes_dist_euc2 = np.load("p46_ShapeDist_6144_Euc.npy")

In [9]:
tree2 = dict()

for i in range(10):
    tree2[i] = dict()
    tree2[i]['child'] = [i+26]
    tree2[i]['parent'] = []

tree2[10] = {'child': [36, 37], 'parent': []}
tree2[11] = {'child': [38, 39], 'parent': []}

for i in range(12,26):
    tree2[i] = dict()
    tree2[i]['child'] = [i+28]
    tree2[i]['parent'] = []

for i in range(26, 36):
    tree2[i] = dict()
    tree2[i]['child'] = [i+28]
    tree2[i]['parent'] = [i-26]

tree2[36] = {'child': [64], 'parent': [10]}
tree2[37] = {'child': [65], 'parent': [10]}
tree2[38] = {'child': [66], 'parent': [11]}
tree2[39] = {'child': [67], 'parent': [11]}

for i in range(40, 43):
    tree2[i] = dict()
    tree2[i]['child'] = [i+28]
    tree2[i]['parent'] = [i-28]

tree2[43] = {'child': [], 'parent': [15]}

for i in range(44, 54):
    tree2[i] = dict()
    tree2[i]['child'] = [i+27]
    tree2[i]['parent'] = [i-28]

for i in range(54, 64):
    tree2[i] = dict()
    tree2[i]['child'] = [i+27]
    tree2[i]['parent'] = [i-28]

tree2[64] = {'child': [91], 'parent': [36]}
tree2[65] = {'child': [], 'parent': [37]}
tree2[66] = {'child': [92], 'parent': [38]}
tree2[67] = {'child': [], 'parent': [39]}

for i in range(68, 71):
    tree2[i] = dict()
    tree2[i]['child'] = [i+25]
    tree2[i]['parent'] = [i-28]

for i in range(71, 81):
    tree2[i] = dict()
    tree2[i]['child'] = [i+25]
    tree2[i]['parent'] = [i-27]

for i in range(81, 91):
    tree2[i] = dict()
    tree2[i]['child'] = [i+25]
    tree2[i]['parent'] = [i-27]

tree2[91] = {'child': [116], 'parent': [64]}
tree2[92] = {'child': [117, 118], 'parent': [66]}

for i in range(93, 104):
    tree2[i] = dict()
    tree2[i]['child'] = [i+26]
    tree2[i]['parent'] = [i-25]

tree2[104] = {'child': [], 'parent': [79]}
tree2[105] = {'child': [130], 'parent': [80]}

for i in range(106, 117):
    tree2[i] = dict()
    tree2[i]['child'] = [i+25]
    tree2[i]['parent'] = [i-25]

tree2[117] = {'child': [142], 'parent': [92]}
tree2[118] = {'child': [143], 'parent': [92]}

for i in range(119, 128):
    tree2[i] = dict()
    tree2[i]['child'] = [i+25]
    tree2[i]['parent'] = [i-26]

tree2[128] = {'child': [], 'parent': [102]}
tree2[129] = {'child': [153], 'parent': [103]}
tree2[130] = {'child': [154], 'parent': [105]}

for i in range(131, 152):
    tree2[i] = dict()
    tree2[i]['child'] = [i+24]
    tree2[i]['parent'] = [i-25]

tree2[152] = {'child': [], 'parent': [127]}
tree2[153] = {'child': [176], 'parent': [129]}
tree2[154] = {'child': [177], 'parent': [130]}

tree2[155] = {'child': [178], 'parent': [131]}
tree2[156] = {'child': [179], 'parent': [132]}
tree2[157] = {'child': [], 'parent': [133]}

tree2[158] = {'child': [180], 'parent': [134]}
tree2[159] = {'child': [181], 'parent': [135]}
tree2[160] = {'child': [], 'parent': [136]}

for i in range(161, 176):
    tree2[i] = dict()
    tree2[i]['child'] = [i+21]
    tree2[i]['parent'] = [i-24]

tree2[176] = {'child': [197], 'parent': [153]}
tree2[177] = {'child': [], 'parent': [154]}

tree2[178] = {'child': [], 'parent': [155]}
tree2[179] = {'child': [], 'parent': [156]}
tree2[180] = {'child': [], 'parent': [158]}
tree2[181] = {'child': [], 'parent': [159]}

for i in range(182, 198):
    tree2[i] = dict()
    tree2[i]['child'] = []
    tree2[i]['parent'] = [i-21]
    
# Verifying if the tree is correct

## Print all leaf nodes
leafs2 = []
for key in tree2:
    if(len(tree2[key]["child"])==0):
        leafs2.append(key)
print("Leaf nodes:", leafs2)

## Print all at the lowest threshold nodes
roots2 = []
for key in tree2:
    if(len(tree2[key]["parent"])==0):
        roots2.append(key)
print("Root nodes:", roots2)

## Print all paths
paths2 = []
for node in leafs2:
    node_path = [node]
    cur_node = node
    while(True):
        if(len(tree2[cur_node]["parent"])!=0):
            node_path.append(tree2[cur_node]["parent"][0])
            cur_node = tree2[cur_node]["parent"][0]
            continue
        else:
            break
    paths2.append(node_path[::-1])
print("Paths:", paths2)

Leaf nodes: [43, 65, 67, 104, 128, 152, 157, 160, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197]
Root nodes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
Paths: [[15, 43], [10, 37, 65], [11, 39, 67], [24, 52, 79, 104], [22, 50, 77, 102, 128], [21, 49, 76, 101, 127, 152], [2, 28, 56, 83, 108, 133, 157], [5, 31, 59, 86, 111, 136, 160], [25, 53, 80, 105, 130, 154, 177], [0, 26, 54, 81, 106, 131, 155, 178], [1, 27, 55, 82, 107, 132, 156, 179], [3, 29, 57, 84, 109, 134, 158, 180], [4, 30, 58, 85, 110, 135, 159, 181], [6, 32, 60, 87, 112, 137, 161, 182], [7, 33, 61, 88, 113, 138, 162, 183], [8, 34, 62, 89, 114, 139, 163, 184], [9, 35, 63, 90, 115, 140, 164, 185], [10, 36, 64, 91, 116, 141, 165, 186], [11, 38, 66, 92, 117, 142, 166, 187], [11, 38, 66, 92, 118, 143, 167, 188], [12, 40, 68, 93, 119, 144, 168, 189], [13, 41, 69, 94, 120, 145, 169, 190], [14, 42, 70, 95, 121, 146, 170, 191]

In [10]:
subchildren2 = get_complete_children_sets(tree2)

## Defining best clusters

In [15]:
best_cluster1_k1 = [123, 164, 194, 65, 163, 188, 125, 218, 244, 159, 68, 167, 193, 63, 245, 145, 186, 148, 49, 48, 52, 233, 183, 171, 140, 201, 203, 204, 205, 109, 238, 239, 155, 196, 137, 170, 207, 45]
best_cluster1_k3 = [156, 123, 238, 239, 185, 154, 155, 196, 223, 197, 212, 116, 150, 205, 233, 115, 146, 45, 142, 201, 199, 225, 231, 228, 229, 163, 191, 245, 65, 158, 190, 92, 164, 167, 134, 165, 193, 63]
best_cluster1_k5 = [245, 193, 63, 134, 167, 163, 165, 164, 65, 223, 171, 169, 225, 196, 173, 174, 228, 204, 185, 212, 238, 239, 182, 116, 180, 208, 145, 205, 146, 45, 154, 155, 188, 125, 158, 159, 123, 191]
best_cluster1_k7 = [196, 171, 169, 198, 225, 188, 158, 123, 155, 154, 125, 174, 205, 203, 204, 226, 145, 149, 180, 208, 146, 45, 212, 238, 239, 182, 185, 163, 245, 65, 191, 159, 167, 164, 165, 134, 193, 63]

In [16]:
best_cluster2_k1 = [97, 73, 42, 71, 41, 189, 92, 67, 116, 65, 137, 185, 139, 43, 152, 48, 125, 78, 79, 22, 105, 162, 160, 58, 180, 133, 55, 106]
best_cluster2_k3 = [74, 92, 67, 185, 73, 123, 43, 41, 71, 121, 189, 141, 65, 152, 48, 77, 79, 105, 78, 155, 107, 133, 158, 139, 162, 159, 161, 86]
best_cluster2_k5 = [97, 149, 101, 50, 74, 147, 43, 176, 105, 79, 189, 191, 69, 140, 141, 65, 142, 167, 67, 139, 137, 138, 31, 159, 158, 131, 179, 133, 100]
best_cluster2_k7 = [176, 79, 77, 101, 100, 105, 131, 132, 158, 138, 137, 31, 159, 133, 139, 115, 141, 65, 142, 167, 39, 149, 74, 97, 147, 43, 191, 189, 145]

## Train on p7 Test on p46

In [20]:
best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_nc2)
best_mlcluster2_nc2_k1 = get_best_cluster_in_other(1, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

## Trying different k values while using best_cluster1_k1 as training set for ML

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k1)
best_mlcluster2_k1_k1 = get_best_cluster_in_other(1, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k1)
best_mlcluster2_k1_k3 = get_best_cluster_in_other(3, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k1)
best_mlcluster2_k1_k5 = get_best_cluster_in_other(5, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k1)
best_mlcluster2_k1_k7 = get_best_cluster_in_other(7, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k1)
best_mlcluster2_k1_k10 = get_best_cluster_in_other(10, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k1)
best_mlcluster2_k1_k15 = get_best_cluster_in_other(15, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

## Trying different k values while using best_cluster1_k3 as training set for ML

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k3)
best_mlcluster2_k3_k1 = get_best_cluster_in_other(1, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k3)
best_mlcluster2_k3_k3 = get_best_cluster_in_other(3, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k3)
best_mlcluster2_k3_k5 = get_best_cluster_in_other(5, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k3)
best_mlcluster2_k3_k7 = get_best_cluster_in_other(7, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k3)
best_mlcluster2_k3_k10 = get_best_cluster_in_other(10, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k3)
best_mlcluster2_k3_k15 = get_best_cluster_in_other(15, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

## Trying different k values while using best_cluster1_k5 as training set for ML

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k5)
best_mlcluster2_k5_k1 = get_best_cluster_in_other(1, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k5)
best_mlcluster2_k5_k3 = get_best_cluster_in_other(3, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k5)
best_mlcluster2_k5_k5 = get_best_cluster_in_other(5, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k5)
best_mlcluster2_k5_k7 = get_best_cluster_in_other(7, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k5)
best_mlcluster2_k5_k10 = get_best_cluster_in_other(10, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k5)
best_mlcluster2_k5_k15 = get_best_cluster_in_other(15, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

## Trying different k values while using best_cluster1_k7 as training set for ML

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k7)
best_mlcluster2_k7_k1 = get_best_cluster_in_other(1, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k7)
best_mlcluster2_k7_k3 = get_best_cluster_in_other(3, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k7)
best_mlcluster2_k7_k5 = get_best_cluster_in_other(5, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k7)
best_mlcluster2_k7_k7 = get_best_cluster_in_other(7, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k7)
best_mlcluster2_k7_k10 = get_best_cluster_in_other(10, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_1, best_cluster1_k7)
best_mlcluster2_k7_k15 = get_best_cluster_in_other(15, roots2, subchildren2, shapes_reps_2, best_cluster_reps, 0)

In [24]:
print("best_mlcluster2_k1_k1", best_mlcluster2_k1_k1)
print("best_mlcluster2_k1_k3", best_mlcluster2_k1_k3)
print("best_mlcluster2_k1_k5", best_mlcluster2_k1_k5)
print("best_mlcluster2_k1_k7", best_mlcluster2_k1_k7)
print("best_mlcluster2_k1_k10", best_mlcluster2_k1_k10)
print("best_mlcluster2_k1_k15", best_mlcluster2_k1_k15)
print()

print("best_mlcluster2_k3_k1", best_mlcluster2_k3_k1)
print("best_mlcluster2_k3_k3", best_mlcluster2_k3_k3)
print("best_mlcluster2_k3_k5", best_mlcluster2_k3_k5)
print("best_mlcluster2_k3_k7", best_mlcluster2_k3_k7)
print("best_mlcluster2_k3_k10", best_mlcluster2_k3_k10)
print("best_mlcluster2_k3_k15", best_mlcluster2_k3_k15)
print()

print("best_mlcluster2_k5_k1", best_mlcluster2_k5_k1)
print("best_mlcluster2_k5_k3", best_mlcluster2_k5_k3)
print("best_mlcluster2_k5_k5", best_mlcluster2_k5_k5)
print("best_mlcluster2_k5_k7", best_mlcluster2_k5_k7)
print("best_mlcluster2_k5_k10", best_mlcluster2_k5_k10)
print("best_mlcluster2_k5_k15", best_mlcluster2_k5_k15)
print()

print("best_mlcluster2_k7_k1", best_mlcluster2_k7_k1)
print("best_mlcluster2_k7_k3", best_mlcluster2_k7_k3)
print("best_mlcluster2_k7_k5", best_mlcluster2_k7_k5)
print("best_mlcluster2_k7_k7", best_mlcluster2_k7_k7)
print("best_mlcluster2_k7_k10", best_mlcluster2_k7_k10)
print("best_mlcluster2_k7_k15", best_mlcluster2_k7_k15)

best_mlcluster2_k1_k1 [155, 132, 133, 180, 110, 59, 182, 61, 184, 90, 141, 65, 166, 167, 67, 168, 41, 191, 43, 192, 123, 149, 195, 175, 152, 50, 176, 24, 80]
best_mlcluster2_k1_k3 [178, 132, 133, 180, 159, 59, 182, 88, 184, 115, 141, 65, 142, 167, 67, 168, 69, 191, 43, 122, 123, 149, 150, 175, 152, 50, 176, 79, 80]
best_mlcluster2_k1_k5 [178, 132, 133, 180, 85, 59, 182, 88, 184, 115, 141, 65, 142, 167, 67, 189, 69, 191, 43, 122, 123, 149, 150, 175, 152, 50, 176, 79, 80]
best_mlcluster2_k1_k7 [131, 132, 133, 180, 85, 59, 182, 162, 184, 115, 141, 65, 142, 167, 67, 189, 69, 191, 43, 122, 123, 149, 150, 175, 152, 50, 176, 79, 80]
best_mlcluster2_k1_k10 [131, 132, 133, 180, 85, 59, 161, 88, 184, 115, 141, 65, 142, 167, 67, 189, 69, 191, 43, 122, 123, 149, 150, 126, 152, 50, 176, 79, 80]
best_mlcluster2_k1_k15 [131, 132, 133, 158, 159, 111, 137, 162, 163, 115, 141, 65, 142, 167, 67, 168, 145, 191, 43, 122, 97, 149, 150, 175, 152, 50, 176, 79, 105]

best_mlcluster2_k3_k1 [131, 132, 133, 180, 

In [21]:
print("\nFor best_cluster2_k1")

print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_cluster2_k1),axis=0), shapes_dist_euc2, 1)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k1_k1),axis=0), shapes_dist_euc2, 1)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k1_k3),axis=0), shapes_dist_euc2, 1)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k1_k5),axis=0), shapes_dist_euc2, 1)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k1_k7),axis=0), shapes_dist_euc2, 1)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k1_k10),axis=0), shapes_dist_euc2, 1)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k1_k15),axis=0), shapes_dist_euc2, 1)[0]))


print("\nFor best_cluster2_k3")

print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_cluster2_k3),axis=0), shapes_dist_euc2, 3)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k3_k1),axis=0), shapes_dist_euc2, 3)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k3_k3),axis=0), shapes_dist_euc2, 3)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k3_k5),axis=0), shapes_dist_euc2, 3)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k3_k7),axis=0), shapes_dist_euc2, 3)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k3_k10),axis=0), shapes_dist_euc2, 3)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k3_k15),axis=0), shapes_dist_euc2, 3)[0]))

print("\nFor best_cluster2_k5")

print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_cluster2_k5),axis=0), shapes_dist_euc2, 5)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k5_k1),axis=0), shapes_dist_euc2, 5)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k5_k3),axis=0), shapes_dist_euc2, 5)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k5_k5),axis=0), shapes_dist_euc2, 5)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k5_k7),axis=0), shapes_dist_euc2, 5)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k5_k10),axis=0), shapes_dist_euc2, 5)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k5_k15),axis=0), shapes_dist_euc2, 5)[0]))

print("\nFor best_cluster2_k7")

print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_cluster2_k7),axis=0), shapes_dist_euc2, 7)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k7_k1),axis=0), shapes_dist_euc2, 7)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k7_k3),axis=0), shapes_dist_euc2, 7)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k7_k5),axis=0), shapes_dist_euc2, 7)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k7_k7),axis=0), shapes_dist_euc2, 7)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k7_k10),axis=0), shapes_dist_euc2, 7)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster2_k7_k15),axis=0), shapes_dist_euc2, 7)[0]))


For best_cluster2_k1
52.062228778436186
73.56131451244971
64.59078194405673
64.4848672384099
64.20333752248177
64.45831470681364
65.87880693869785

For best_cluster2_k3
76.70951441943777
88.1973987129151
87.42110603426546
86.60392519617605
84.79929075672975
84.08062853970256
83.99363879372159

For best_cluster2_k5
92.96997269774653
106.5866831096719
101.50345002806291
101.41726126641356
99.12526880575079
98.57442022068948
97.46449898019057

For best_cluster2_k7
104.66908636346488
119.80988594597909
114.76022049093163
113.8097984384725
112.36348052318417
110.3769999635636
109.77636985587152


## Train on p46 Test on p7

In [22]:
## Trying different k values while using best_cluster2_k1 as training set for ML

best_cluster_reps = get_shape_reps(shapes_reps_2, best_cluster2_k1)
best_mlcluster1_k1_k1 = get_best_cluster_in_other(1, roots1, subchildren1, shapes_reps_1, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_2, best_cluster2_k1)
best_mlcluster1_k1_k3 = get_best_cluster_in_other(3, roots1, subchildren1, shapes_reps_1, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_2, best_cluster2_k1)
best_mlcluster1_k1_k5 = get_best_cluster_in_other(5, roots1, subchildren1, shapes_reps_1, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_2, best_cluster2_k1)
best_mlcluster1_k1_k7 = get_best_cluster_in_other(7, roots1, subchildren1, shapes_reps_1, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_2, best_cluster2_k1)
best_mlcluster1_k1_k10 = get_best_cluster_in_other(10, roots1, subchildren1, shapes_reps_1, best_cluster_reps, 0)

best_cluster_reps = get_shape_reps(shapes_reps_2, best_cluster2_k1)
best_mlcluster1_k1_k15 = get_best_cluster_in_other(15, roots1, subchildren1, shapes_reps_1, best_cluster_reps, 0)

In [25]:
print("best_mlcluster1_k1_k1", best_mlcluster1_k1_k1)
print("best_mlcluster1_k1_k3", best_mlcluster1_k1_k3)
print("best_mlcluster1_k1_k5", best_mlcluster1_k1_k5)
print("best_mlcluster1_k1_k7", best_mlcluster1_k1_k7)
print("best_mlcluster1_k1_k10", best_mlcluster1_k1_k10)
print("best_mlcluster1_k1_k15", best_mlcluster1_k1_k15)

best_mlcluster1_k1_k1 [70, 169, 223, 139, 74, 141, 142, 228, 204, 230, 231, 80, 45, 179, 115, 83, 84, 212, 238, 239, 87, 154, 89, 123, 124, 57, 58, 159, 191, 129, 162, 63, 131, 65, 132, 165, 68, 167]
best_mlcluster1_k1_k3 [34, 169, 223, 139, 225, 201, 142, 203, 204, 205, 145, 80, 45, 179, 82, 83, 84, 212, 86, 87, 121, 54, 123, 124, 157, 58, 159, 191, 129, 62, 63, 131, 65, 132, 165, 68, 167]
best_mlcluster1_k1_k5 [34, 169, 223, 139, 200, 201, 174, 203, 204, 205, 145, 80, 45, 179, 82, 83, 84, 212, 86, 87, 121, 54, 123, 124, 92, 58, 159, 191, 129, 97, 63, 131, 65, 132, 100, 68, 167]
best_mlcluster1_k1_k7 [136, 169, 198, 139, 225, 201, 174, 203, 204, 205, 145, 80, 45, 179, 115, 83, 84, 85, 119, 87, 53, 54, 123, 124, 92, 126, 159, 191, 129, 97, 63, 131, 65, 132, 100, 68, 167]
best_mlcluster1_k1_k10 [136, 169, 198, 139, 200, 201, 174, 203, 204, 205, 145, 113, 45, 114, 115, 83, 84, 183, 119, 87, 53, 155, 123, 91, 92, 126, 159, 191, 96, 97, 63, 131, 65, 66, 100, 68, 167]
best_mlcluster1_k1_k15

In [23]:
print("\nFor best_cluster1_k1")

print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_cluster1_k1),axis=0), shapes_dist_euc1, 1)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster1_k1_k1),axis=0), shapes_dist_euc1, 1)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster1_k1_k3),axis=0), shapes_dist_euc1, 1)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster1_k1_k5),axis=0), shapes_dist_euc1, 1)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster1_k1_k7),axis=0), shapes_dist_euc1, 1)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster1_k1_k10),axis=0), shapes_dist_euc1, 1)[0]))
print(1/(get_fitness_KnearestNeighbour(np.expand_dims(np.array(best_mlcluster1_k1_k15),axis=0), shapes_dist_euc1, 1)[0]))


For best_cluster1_k1
36.65190519863082
47.46419693877804
44.60676632207685
45.41308340019728
47.2328730314108
45.16523681577102
49.31404712036
